In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import detrend

def filter_eeg_bandpass(eeg, fs, lowcut=0.2, highcut=30, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return signal.filtfilt(b, a, eeg)


def compute_standard_spectrogram(eeg_raw, fs, epoch_sec=0.5, step_sec=0.1,
                                 fmin=0.2, fmax=48, log_scale=True):
    """
    Compute a Welch-based time-frequency representation using sliding epochs.
    Parameters
    ----------
    eeg_raw : 1D array
        Input EEG/LFP signal.
    fs : int
        Sampling rate in Hz.
    epoch_sec : float
        Window length in seconds.
    step_sec : float
        Step size in seconds.
    fmin, fmax : float
        Frequency range to keep.
    log_scale : bool
        Whether to return log power.

    Returns
    -------
    freqs_sel : 1D array
        Selected frequencies.
    t_centers : 1D array
        Center time of each epoch, in seconds.
    tf : 2D array
        Shape = (n_freqs, n_epochs)
    """
    eeg_raw = np.asarray(eeg_raw).squeeze()
    nper = int(epoch_sec * fs)
    step = int(step_sec * fs)

    if nper <= 0 or step <= 0:
        raise ValueError("epoch_sec and step_sec must be positive.")
    if len(eeg_raw) < nper:
        raise ValueError("Signal is shorter than one epoch.")

    starts = np.arange(0, len(eeg_raw) - nper + 1, step)
    t_centers = (starts + nper / 2) / fs

    tf_list = []
    freqs_sel = None

    for s in starts:
        seg = eeg_raw[s:s + nper]

        # Welch inside each epoch
        nperseg = min(nper, max(64, nper // 2))
        noverlap = nperseg // 2

        freqs, psd = signal.welch(
            seg,
            fs=fs,
            window='hamming',
            nperseg=nperseg,
            noverlap=noverlap
        )

        mask = (freqs >= fmin) & (freqs <= fmax)
        if freqs_sel is None:
            freqs_sel = freqs[mask]

        power = psd[mask]
        if log_scale:
            power = 10 * np.log10(power + 1e-12)

        tf_list.append(power)

    tf = np.stack(tf_list, axis=1)  # (freq, time)
    return freqs_sel, t_centers, tf

In [5]:
# File Path
all_freqs=[]
all_psd=[]
i_values = list(range(10))
rand_seeds = [2,3,4,5,6]

for i in i_values:
    psd_for_this_j = []
    for j in rand_seeds:
        filedir = "../Simulations/"
        filepath = f"{filedir}rand{j}/rand_bio{i}/out/"
        #filepath = f"{filedir}rand{j}/rand_hard{i}/out/"

        # Read data
        time_cx = np.loadtxt(filepath + "time_cx")
        index = np.arange(10000+30000, 10000+30000+5000) #during consolidation
        #index = np.arange(10000, 10000+5000)  #baseline
        time = (index - index[0]) / 1000 # seconds,start at 0
        data = time_cx[index, 1:-1].T

        from scipy.signal import decimate
        bio_current = np.loadtxt(filepath + "bio_current.dat")
        Iext = bio_current[index,1]
        fs = 1000

        # Compute LFP
        LFP = np.sum(data, axis=0) / data.shape[0]
        LFP = detrend(LFP)

        ###psd
        freqs, psd = signal.welch(
            LFP,
            fs=fs,
            window='hann',
            nperseg=fs*2,
            noverlap=fs/2,
        )
        psd_for_this_j.append(psd)
      
        #psd_db = 10 * np.log10(psd + 1e-12)
        #fig, ax = plt.subplots(1, 1, figsize=(3,2.5), sharex=False)
        #ax.plot(freqs, psd_db, 'k', linewidth=2)
        #ax.spines['top'].set_visible(False)
        #ax.spines['right'].set_visible(False)
        #ax.set_xlabel("Frequency (Hz)")
        #ax.set_ylabel("Power (dB)")
        #ax.set_xlim([0,60])
        #ax.set_ylim([-50,0])
        #plt.tight_layout()
        #plt.show()
        #fig.savefig(filepath+'psd.svg', bbox_inches='tight')
        #fig.savefig(filepath+'psd.png', bbox_inches='tight')
       
        
        np.savez(filepath+'psd_data.npz', freqs=freqs, psd=psd, time=time, LFP=LFP)
    all_psd.append(psd_for_this_j)

# All Welch calculations use the same frequency axis
all_freqs = freqs
all_psd = np.asarray(all_psd)    
np.savez(filedir+'psd_data_bio.npz', i_values=np.asarray(i_values), rand_seeds=np.asarray(rand_seeds), freqs=all_freqs, psd=all_psd)